In [1]:
# Connect and Load Raw Data
import pandas as pd
import json
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://localhost\\SQLEXPRESS/TMDB?"
    "driver=ODBC+Driver+17+for+SQL+Server&"
    "trusted_connection=yes&"
    "TrustServerCertificate=yes"
)

movies = pd.read_sql("SELECT id, genres, keywords, production_companies FROM dbo.movies", engine)
credits = pd.read_sql("SELECT movie_id, cast, crew FROM dbo.credits", engine)

In [2]:
# Helper Function to Flatten JSON List Columns
def flatten_column(df, id_col, json_col, out_cols=("id", "name")):
    rows = []
    for _, row in df.iterrows():
        try:
            items = json.loads(row[json_col])
        except (ValueError, TypeError):
            continue
        for item in items:
            rows.append({
                "movie_id": row[id_col],
                out_cols[0]: item.get("id"),
                out_cols[1]: item.get("name")
            })
    return pd.DataFrame(rows)

In [3]:
# Flatten Genres, Keywords, Production Companies
genres_flat = flatten_column(movies, "id", "genres")
keywords_flat = flatten_column(movies, "id", "keywords")
companies_flat = flatten_column(movies, "id", "production_companies")

print("genres:", len(genres_flat))
print("keywords:", len(keywords_flat))
print("companies:", len(companies_flat))

genres: 12160
keywords: 36194
companies: 13677


In [4]:
# Flatten Cast
def flatten_cast(df):
    rows = []
    for _, row in df.iterrows():
        try:
            cast_list = json.loads(row["cast"])
        except (ValueError, TypeError):
            continue
        for member in cast_list:
            rows.append({
                "movie_id": row["movie_id"],
                "cast_id": member.get("cast_id"),
                "actor_name": member.get("name"),
                "character": member.get("character"),
                "billing_order": member.get("order")
            })
    return pd.DataFrame(rows)

cast_flat = flatten_cast(credits)
print("cast rows:", len(cast_flat))

cast rows: 106257


In [5]:
# Flatten Crew (Directors, Writers, Producers, etc.)
def flatten_crew(df):
    rows = []
    for _, row in df.iterrows():
        try:
            crew_list = json.loads(row["crew"])
        except (ValueError, TypeError):
            continue
        for member in crew_list:
            rows.append({
                "movie_id": row["movie_id"],
                "name": member.get("name"),
                "job": member.get("job"),
                "department": member.get("department")
            })
    return pd.DataFrame(rows)

crew_flat = flatten_crew(credits)
print("crew rows:", len(crew_flat))

crew rows: 129581


In [6]:
# Write Flattened Tables to SQL Server
genres_flat.to_sql("genres", engine, if_exists="replace", index=False)
keywords_flat.to_sql("keywords_flat", engine, if_exists="replace", index=False)
companies_flat.to_sql("production_companies_flat", engine, if_exists="replace", index=False)
cast_flat.to_sql("cast_members", engine, if_exists="replace", index=False)
crew_flat.to_sql("crew_members", engine, if_exists="replace", index=False)

print("All flattened tables written to SQL Server.")

All flattened tables written to SQL Server.
